# LAB 03 — Delta DML & Time Travel

> **Ticket INC-118** · From: Sales Operations · Priority: High
>
> Sales Operations sent today's customer update file. Merge it into `bronze.customers`: existing customers get the new values and new customers are added. The Austin records in the file carry the wrong state and must be corrected.

**Definition of done**

Ticket INC-118:
- Task 1: the view `v_updates` holds the rows of `customers_new.csv` and has the same columns as `bronze.customers`.
- Task 2: one statement updated the matching customers and inserted the new ones. It read every row of the update file and deleted none.
- Task 3: every Austin customer has state `'TX'`, and no other customer changed.

Alert INC-119 (it comes in at Task 4):
- Task 4: the change behind the alert is in the history of `bronze.customers`.
- Task 5: the history shows the upsert of Task 2 and the change that removed the rows.
- Task 6: the view `v_recovered` reads the table as it was right before that change and has as many rows as were lost.
- Task 7: the table is back at a version from before the loss (after the state fix of Task 3) and has as many rows as were lost.
- Task 8: the old data files are removed with no retention, the view `v_version0` records that time travel to version 0 is rejected, and the current table can still be read.

A task is done when its validation cell prints `Task N OK`. For Tasks 2, 3, 4, 7 and 8a, **Check your work** in the app must pass as well. Task 4 has no validation cell, so **Check your work** alone finishes it.

Work in the Workshop Guide app in the left window, and paste the code into the marked cells of this notebook.

Every task has a SQL track and a PySpark track in the app. Both tracks change the same table and create the same temporary views, so the validation cells work with either track.

**Prerequisites:** Attach the notebook to **Serverless** compute. The lab needs only `00_setup`. You do not need the LAB 02 tables, because the Setup cells rebuild `bronze.customers` from `customers.csv`.

| Task | What you do |
|------|-------------|
| Task 1 | Load the update file `customers_new.csv` into the view `v_updates` |
| Task 2 | Upsert `v_updates` into the customers table (update existing customers, add new ones) |
| Task 3 | Correct the state of the Austin customers to `'TX'` |
| Task 4 | Run the provided cell (no code to write) and read the alert |
| Task 5 | Find in the table's version log what removed the rows |
| Task 6 | Read the version from before the rows were lost |
| Task 7 | Bring the table back to that version |
| Task 8 | Remove the old data files (8a), then check that time travel to version 0 fails (8b) |

## Setup

Run both cells. The second one rebuilds `bronze.customers` from `customers.csv` and drops the views of an earlier attempt. Run the cells again only if you want to start over.

In [0]:
%run ../setup/00_setup

In [0]:
# Rebuild bronze.customers from customers.csv and drop the views of an earlier attempt
customers_path = f"{DATASET_PATH}/customers/customers.csv"
df_base = spark.read.format("csv").option("header", True).option("inferSchema", True).load(customers_path)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.customers")
df_base.write.mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers")
# Start over: drop the views of an earlier attempt
for view in ("v_updates", "v_recovered", "v_version0"):
    spark.sql(f"DROP VIEW IF EXISTS {view}")
print(f"Base table ready: {spark.table(f'{CATALOG}.{BRONZE_SCHEMA}.customers').count()} rows")

## Task 1 — Examine the Update File

Look at an update file before you merge it into a table. Its first line holds the column names, and its columns must match `bronze.customers`, so that the merge in Task 2 can take every column as it is. The cell reads `customers_new.csv`, registers its rows as the temporary view `v_updates` and shows the first rows. Get the code from **Task 1** in the Workshop Guide.

**Expected:** 14 update rows (the base table has 10,000).

In [0]:
# SQL run from Python: the f-string puts in the volume path DATASET_PATH
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_updates AS
SELECT * EXCEPT (_rescued_data)
FROM read_files(
  '{DATASET_PATH}/customers/customers_new.csv',
  format => 'csv',
  header => true
)
""")

display(spark.sql("SELECT * FROM v_updates LIMIT 5"))

In [0]:
# -- Validation --
updates = spark.table("v_updates")
file_rows = spark.read.format("csv").option("header", True).load(f"{DATASET_PATH}/customers/customers_new.csv").count()
assert updates.count() == file_rows, f"v_updates has {updates.count()} rows, customers_new.csv has {file_rows}: read the first line as column names"
expected_columns = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").dtypes
assert updates.dtypes == expected_columns, f"The columns of v_updates differ from bronze.customers: {updates.dtypes}"
print(f"Task 1 OK: {file_rows} update rows in v_updates, same columns as bronze.customers")

## Task 2 — Upsert the Update File

An upsert applies the whole update file in one atomic operation. Customers that already exist get the new values, new customers are added, and no customer is removed. You use it when a source system sends changed and new rows in the same file. The cell merges `v_updates` from Task 1 into `bronze.customers`. Get the code from **Task 2** in the Workshop Guide.

**Expected:** 10,006 rows after the upsert. 8 rows of the update file match existing customers, and 6 are new. One matching customer ID appears twice in `bronze.customers`, so the upsert updates 9 rows, and the validation cell reports `updated 9`.

In [0]:
%sql
MERGE INTO bronze.customers AS t
USING v_updates AS s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT count(*) AS customers_after_merge FROM bronze.customers

In [0]:
# -- Validation --
history = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers").collect()
merge = next((r for r in history if r["operation"] == "MERGE"), None)
assert merge, "No MERGE in the history of bronze.customers: paste the Task 2 code into its cell and run it"
m = merge["operationMetrics"]
assert m.get("numTargetRowsDeleted", "0") == "0", f"The MERGE deleted {m['numTargetRowsDeleted']} customers: leave out the clause that deletes"
assert int(m["numTargetRowsUpdated"]) > 0 and int(m["numTargetRowsInserted"]) > 0, \
    f"The MERGE updated {m['numTargetRowsUpdated']} and inserted {m['numTargetRowsInserted']} rows: it must update existing customers and insert new ones"
total = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
print(f"Task 2 OK: MERGE updated {m['numTargetRowsUpdated']} and inserted {m['numTargetRowsInserted']} rows, {total} customers now")

## Task 3 — Correct the Austin Records

Correct the `state` of the Austin customers to `'TX'`. An update changes the rows in place, and its condition decides which rows change. No other row may change.

> **Note:** The base file has no Austin customers. The Austin rows come with the update file that you merged in Task 2, and they carry the wrong states (`CA`, `MI`). This task fixes them. Expected: **5** Austin rows after the upsert. The file has 4, and one of those IDs appears twice in the base table, so the upsert updates both rows.

Get the code from **Task 3** in the Workshop Guide.

In [0]:
%sql
UPDATE bronze.customers
SET state = 'TX'
WHERE city = 'Austin';

SELECT * FROM bronze.customers WHERE city = 'Austin'

In [0]:
# -- Validation --
austin = spark.sql(f"SELECT state FROM {CATALOG}.{BRONZE_SCHEMA}.customers WHERE city = 'Austin'").collect()
assert len(austin) > 0, "No Austin customers found — did the MERGE in Task 2 run?"
wrong = [r["state"] for r in austin if r["state"] != "TX"]
assert not wrong, f"Some Austin customers still have a state other than 'TX': {wrong}"
print(f"Task 3 OK: {len(austin)} Austin customers, all with state = 'TX'")


## Task 4 — Alert INC-119 (run only, no code to write)

> **Alert INC-119** · Priority: Critical
>
> `bronze.customers` lost rows a few minutes ago. Find out what happened, bring the data back and tell us how long a restore like this stays possible.

The next cell runs the cleanup script behind the alert. Its condition is true for every customer, so it removes every row. Run the cell as it is, then select **Check your work** for Task 4 in the Workshop Guide. In Tasks 5–8 you answer the alert.

In [0]:
# Alert INC-119: run the cleanup script that deletes the rows of bronze.customers
count_before = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
print(f"Rows BEFORE delete: {count_before}")

# "Accident" - deletes every row (no customer has a NULL country)
spark.sql(f"""
    DELETE FROM {CATALOG}.{BRONZE_SCHEMA}.customers
    WHERE country IS NOT NULL
""")

count_after = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
print(f"Rows AFTER delete: {count_after} (lost {count_before - count_after} rows!)")

## Task 5 — Find What Removed the Rows

Every write to a Delta table adds a version to its transaction log. When data changes and you do not know why, look at the table versions first. One command lists each version with its operation, parameters and metrics. Get the code from **Task 5** in the Workshop Guide.

In the output, find the version that removed the rows. On serverless you may also see **OPTIMIZE** commits that you did not run. They come from automatic compaction after a write: they rewrite files but do not change the data.

In [0]:
%sql
DESCRIBE HISTORY bronze.customers

In [0]:
# -- Validation --
history = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers").collect()
operations = [row["operation"] for row in history]
assert "DELETE" in operations, "Expected DELETE in history — did you run the Task 4 cell?"
assert "MERGE" in operations, "Expected MERGE in history — did your MERGE in Task 2 run on this table?"
print(f"Task 5 OK: {len(history)} versions found. Operations: {operations}")

## Task 6 — Time Travel: Read the Version Before the Loss

Time travel reads an earlier version of a table without changing it. Use it to see the data as it was before a bad change, before you repair anything. The cell reads the table as it was right before the change behind the alert and registers the rows as the temporary view `v_recovered`. That version is the state fix of Task 3, or on serverless often an automatic **OPTIMIZE** commit with the same data. Get the code from **Task 6** in the Workshop Guide.

**Expected:** `v_recovered` has 10,006 rows, and the current table has 0.

In [ ]:
# The version right before the DELETE, read from the history (the row right below the DELETE in Task 5)
history = spark.sql("DESCRIBE HISTORY bronze.customers")
version_before_delete = history.where("operation = 'DELETE'").selectExpr("max(version)").first()[0] - 1

# SQL run from Python: the f-string puts the version into the SQL
spark.sql(f"CREATE OR REPLACE TEMP VIEW v_recovered AS SELECT * FROM bronze.customers VERSION AS OF {version_before_delete}")

display(spark.sql(f"""
SELECT
  {version_before_delete} AS version_read,
  (SELECT count(*) FROM v_recovered) AS recovered_rows,
  (SELECT count(*) FROM bronze.customers) AS current_rows
"""))

In [0]:
# -- Validation --
history = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers").collect()
delete = next((r for r in history if r["operation"] == "DELETE"), None)
assert delete, "No DELETE in the history: run the Task 4 cell"
deleted = int(delete["operationMetrics"]["numDeletedRows"])
recovered = spark.table("v_recovered").count()
assert recovered == deleted, f"v_recovered has {recovered} rows, the DELETE removed {deleted}: read the table as it was right before the DELETE"
print(f"Task 6 OK: v_recovered has {recovered} rows, as many as the DELETE removed")

## Task 7 — Bring the Table Back

Reading the old version in Task 6 does not repair the table. One command makes an earlier version the current state again. It adds a new entry to the history, so the history keeps both the loss and the repair. Bring back the version that you read in Task 6. Get the code from **Task 7** in the Workshop Guide.

**Expected:** 10,006 rows, the same count as before the loss.

In [ ]:
# The version right before the DELETE, read from the history (the same version as in Task 6)
history = spark.sql("DESCRIBE HISTORY bronze.customers")
version_before_delete = history.where("operation = 'DELETE'").selectExpr("max(version)").first()[0] - 1

# SQL run from Python: the f-string puts the version into the SQL
spark.sql(f"RESTORE TABLE bronze.customers TO VERSION AS OF {version_before_delete}")

display(spark.sql("SELECT count(*) AS rows_after_restore FROM bronze.customers"))

In [0]:
# -- Validation --
history = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers").collect()
delete = next(r for r in history if r["operation"] == "DELETE")
restore = next((r for r in history if r["operation"] == "RESTORE" and r["version"] > delete["version"]), None)
assert restore, "No RESTORE after the DELETE in the history: paste the Task 7 code into its cell and run it"
deleted = int(delete["operationMetrics"]["numDeletedRows"])
current = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
assert current == deleted, f"bronze.customers has {current} rows, the DELETE removed {deleted}: restore the version right before the DELETE"
print(f"Task 7 OK: RESTORE (version {restore['version']}) brought back {current} rows, as many as the DELETE removed")

## Task 8 — VACUUM and Its Impact on Time Travel

The alert also asks how long a restore like this stays possible. Time travel needs the data files of the old version. `VACUUM` deletes the files that the current version no longer uses and that are older than the retention period. Delta also refuses to read versions older than `delta.deletedFileRetentionDuration`, because VACUUM may have deleted their files. On Unity Catalog managed tables the physical deletion may happen later, so what you see here is the retention guard.

> **Note:** Removing the old files with no retention is for this lab only. The default retention is **7 days**. Do not lower it in production unless you understand the consequences.

1. Run the provided cell. It prepares the table so that VACUUM can remove the old files.
2. Get the VACUUM code from **Task 8a** in the Workshop Guide. Paste it into the cell `# Task 8a` and run it.
3. Get the time-travel query from **Task 8b** in the Workshop Guide. Paste it into the cell `# Task 8b` and run it, then run the validation cell.

In [0]:
# Step 1: Check versions available before VACUUM
history_before = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers").collect()
print(f"Versions available: {len(history_before)}")
print(f"Version numbers: {[r['version'] for r in history_before]}")

# Step 2: OPTIMIZE — rewrite the data files so the files of old versions are no longer
# referenced by the current version (needed when deletion vectors are enabled).
# 0 files added and removed is possible: then there was nothing to compact
display(spark.sql(f"OPTIMIZE {CATALOG}.{BRONZE_SCHEMA}.customers").select("metrics.numFilesAdded", "metrics.numFilesRemoved"))

# Step 3: Bypass the retention safety check (LAB ONLY — never do this in production!)
try:
    # Classic compute
    spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
    print("Retention safety check disabled via Spark conf (classic compute)")
except Exception as e:
    # Serverless: conf not available (CONFIG_NOT_AVAILABLE) -> lower the table's own retention instead
    spark.sql(f"ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.customers SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')")
    print(f"Spark conf not settable here ({type(e).__name__}) — set delta.deletedFileRetentionDuration = 0 hours on the table")


In [0]:
%sql
VACUUM bronze.customers
RETAIN 0 HOURS

In [0]:
# Python on both tracks: a %sql cell stops at the error that this task expects
try:
    spark.sql("SELECT count(*) AS rows_in_version_0 FROM bronze.customers VERSION AS OF 0").collect()
    rejected = False
    print("Unexpected: version 0 is still readable after VACUUM")
except Exception as e:
    rejected = True
    print(f"Version 0 is rejected: {type(e).__name__}: {str(e).splitlines()[0][:200]}")

spark.createDataFrame([(rejected,)], "rejected boolean").createOrReplaceTempView("v_version0")

In [0]:
# -- Validation --
try:
    rejected = spark.table("v_version0").first()["rejected"]
except Exception:
    rejected = None
assert rejected is not None, "v_version0 not found: paste the Task 8b code into its cell and run it"
assert rejected, ("v_version0 says version 0 is still readable: run the provided Task 8 cell (OPTIMIZE + retention), "
                  "then Task 8a (VACUUM with no retention), then Task 8b again")
# Version 0 must fail on the retention guard (or on missing files)
try:
    spark.read.option("versionAsOf", 0).table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
    error = ""
except Exception as e:
    error = str(e)
assert any(k in error for k in ("DELTA_UNSUPPORTED_TIME_TRAVEL_BEYOND_DELETED_FILE_RETENTION_DURATION", "FILE_NOT_FOUND", "FileNotFound")), \
    f"Version 0 is not rejected by the retention guard: {error[:200] or 'it can still be read'}"
# The table property alone already blocks time travel: the history must show VACUUM ... RETAIN 0 HOURS after the RESTORE
history = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers").collect()
restore_version = max((r["version"] for r in history if r["operation"] == "RESTORE"), default=-1)
vacuums = [r for r in history if r["operation"] == "VACUUM START" and r["version"] > restore_version]
assert vacuums, "No VACUUM after the RESTORE in the table history: run the Task 8a cell"
assert any((r["operationParameters"] or {}).get("specifiedRetentionMillis") == "0" for r in vacuums), \
    "VACUUM ran without a retention of 0 hours: check the Task 8a code"
# After VACUUM, the latest version should still be accessible
current = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
assert current > 0, "Current version should still work after VACUUM"
print(f"Task 8 OK: VACUUM with 0 hours retention is in the history, time travel to version 0 is rejected, current table has {current} rows")
print("Key takeaway: VACUUM deletes old unreferenced files → Time Travel to vacuumed versions is rejected (retention guard)")
print(f"  Default retention: 7 days | Production best practice: never set to 0 hours")

## Summary

In this lab you:
- used MERGE for an upsert (insert and update), in SQL or with the `DeltaTable` API
- changed rows with UPDATE and removed them with DELETE
- read the history with `DESCRIBE HISTORY` or `history()`
- queried an earlier version with time travel
- restored a table with `RESTORE TABLE` or `restoreToVersion()`
- ran VACUUM and saw how it limits time travel

Key takeaways:
- VACUUM deletes old data files that the current version no longer uses. After that, Delta rejects time travel to the vacuumed versions.
- The default retention is 7 days. Never set it to 0 hours in production.

> **Exam tip:** Time travel uses the Delta transaction log and the data files that each version references. Only `VACUUM` removes the data files of old versions. The default file retention is **7 days** (`delta.deletedFileRetentionDuration`). Log entries are kept for `delta.logRetentionDuration` (default 30 days). After VACUUM, `DESCRIBE HISTORY` still shows the metadata of old versions, but a query on a vacuumed version fails. Delta rejects versions older than the file retention because their files may be gone. On Unity Catalog managed tables the physical deletion can happen later.

> **Next:** LAB 04 — Streaming & Auto Loader

## Cleanup (Optional)

In [0]:
# Optional cleanup
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.customers")
print("LAB 03 complete.")

← <a href="$../day1/demo/03_delta_lake">03 — Delta Lake</a> | **<a href="$../..">Course folder (README)</a>** | <a href="$../day1/demo/04_incremental_ingestion">04 — Incremental Ingestion →</a>